# XLM-R Base — LODO Only

Binary security detection exactly as in the supplied `xlmr-base.ipynb`:

- `ALLOW = 0`
- `REVIEW / BLOCK = 1`
- Backbone: `FacebookAI/xlm-roberta-base`
- Seeds: `42, 43, 44`

LODO folds:

1. Healthcare + E-commerce → Banking
2. Banking + E-commerce → Healthcare
3. Banking + Healthcare → E-commerce

The held-out target domain is excluded from both training and validation.

`SMOKE_RUN=True` runs all three LODO folds using seed 42, one epoch, and small
subsets. After it passes, set `SMOKE_RUN=False` for the full 3×3 LODO run.

In [ ]:
# ============================================================
# CELL 1 — INSTALL + CONFIG
# ============================================================
%pip install -q -U "transformers>=4.45" "datasets>=3.0" "accelerate>=1.0"
%pip install -q scikit-learn

import gc
import inspect
import math
import random
import shutil
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

MODEL_NAME = "FacebookAI/xlm-roberta-base"

# ---------------- Run mode ----------------
# First run with True. After all 3 LODO folds complete successfully,
# change only this to False for the full experiment.
SMOKE_RUN = True

FULL_SEEDS = [42, 43, 44]
SMOKE_SEEDS = [42]
SEEDS = SMOKE_SEEDS if SMOKE_RUN else FULL_SEEDS

SMOKE_TRAIN_SCENARIOS = 96
SMOKE_VAL_SCENARIOS = 48
SMOKE_TEST_SCENARIOS = 36
SMOKE_EPOCHS = 1

TARGET_DOMAINS = ["banking", "healthcare", "e-commerce"]
TEXT_FORMS = ["bangla", "banglish", "code_mix", "english"]

MAX_SEQ_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 3
ACTIVE_EPOCHS = SMOKE_EPOCHS if SMOKE_RUN else NUM_TRAIN_EPOCHS
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE = 2

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
TEMP_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("./temp")

RUN_MODE = "smoke" if SMOKE_RUN else "full"
OUTPUT_DIR = WORK_ROOT / "xlmr_base_lodo_only" / RUN_MODE
PRED_DIR = OUTPUT_DIR / "predictions"
METRIC_DIR = OUTPUT_DIR / "metrics"
CKPT_DIR = OUTPUT_DIR / "checkpoints"
TEMP_DIR = TEMP_ROOT / "xlmr_base_lodo_only"

for d in [OUTPUT_DIR, PRED_DIR, METRIC_DIR, CKPT_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    warnings.warn("GPU is not enabled.")

print("Mode:", RUN_MODE)
print("Seeds:", SEEDS)
print("Epochs:", ACTIVE_EPOCHS)
print("Targets:", TARGET_DOMAINS)

In [ ]:
# ============================================================
# CELL 2 — LOAD DATA
# ============================================================
SEARCH_ROOTS = [Path("/kaggle/input"), Path(".")]

REQUIRED_COLUMNS = {
    "id", "domain", "security_intent", "hard_negative_type",
    "decision", "risk_level", "code_mix", "banglish",
    "bangla", "english", "scenario_family_id", "split",
}

def locate_split(candidates, required=True):
    hits = []

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for name in candidates:
            hits.extend([
                p for p in root.rglob(name)
                if "__MACOSX" not in p.parts
            ])

    hits = sorted(set(hits))

    if hits:
        preferred = [p for p in hits if "splits_v2" in str(p)]
        return preferred[0] if preferred else hits[0]

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        for z in root.rglob("splits_v2.zip"):
            extract_root = TEMP_DIR / "dataset_extract"
            extract_root.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(z) as archive:
                names = archive.namelist()

                for candidate in candidates:
                    for member in [candidate, f"splits_v2/{candidate}"]:
                        if member in names:
                            archive.extract(member, extract_root)
                            return extract_root / member

    if required:
        raise FileNotFoundError(f"Could not find {candidates}")

    return None

def load_split(path, expected_split=None):
    df = pd.read_csv(path)

    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns: {sorted(missing)}")

    if expected_split is not None:
        assert df["split"].eq(expected_split).all()

    return df

TRAIN_PATH = locate_split(["train.csv", "training.csv"])
VAL_PATH = locate_split(
    ["val.csv", "validation.csv", "valid.csv", "dev.csv"],
    required=False,
)
TEST_PATH = locate_split(["heldout_test.csv"])

FULL_TRAIN = load_split(TRAIN_PATH)
FULL_VAL = load_split(VAL_PATH) if VAL_PATH is not None else None
FULL_TEST = load_split(TEST_PATH, "heldout_test")

if "in_decision_matched_test" not in FULL_TEST.columns:
    FULL_TEST["in_decision_matched_test"] = False

if FULL_TEST["in_decision_matched_test"].dtype != bool:
    FULL_TEST["in_decision_matched_test"] = (
        FULL_TEST["in_decision_matched_test"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )

print("Train:", len(FULL_TRAIN))
print("Val  :", len(FULL_VAL) if FULL_VAL is not None else "derived from source domains")
print("Test :", len(FULL_TEST))

In [ ]:
# ============================================================
# CELL 3 — BUILD STRICT LODO FOLDS
# ============================================================
def source_only_validation_split(df, seed=42):
    groups = (
        df["scenario_family_id"]
        .fillna(df["id"])
        .astype(str)
    )

    strata = (
        df["domain"].astype(str)
        + "::"
        + df["decision"].astype(str)
    )

    try:
        splitter = StratifiedGroupKFold(
            n_splits=5,
            shuffle=True,
            random_state=seed,
        )
        tr_idx, va_idx = next(
            splitter.split(df, y=strata, groups=groups)
        )
    except ValueError:
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.20,
            random_state=seed,
        )
        tr_idx, va_idx = next(
            splitter.split(df, groups=groups)
        )

    return (
        df.iloc[tr_idx].reset_index(drop=True),
        df.iloc[va_idx].reset_index(drop=True),
    )


def smoke_sample(df, n, seed):
    if len(df) <= n:
        return df.copy().reset_index(drop=True)

    work = df.copy()
    work["_stratum"] = work["decision"].astype(str)

    parts = []
    rng = np.random.default_rng(seed)
    groups = list(work.groupby("_stratum", sort=False))
    per = max(1, n // max(len(groups), 1))

    for _,g in groups:
        parts.append(
            g.sample(
                min(per, len(g)),
                random_state=int(rng.integers(0, 2**31-1)),
            )
        )

    out = (
        pd.concat(parts)
        .drop(columns="_stratum")
        .drop_duplicates("id")
    )

    if len(out) < n:
        remaining = df[~df["id"].isin(out["id"])]
        if len(remaining):
            out = pd.concat([
                out,
                remaining.sample(
                    min(n-len(out), len(remaining)),
                    random_state=seed,
                ),
            ])

    return out.head(n).reset_index(drop=True)

def build_lodo_fold(target_domain):
    source_domains = [
        d for d in TARGET_DOMAINS
        if d != target_domain
    ]

    source_train = FULL_TRAIN[
        FULL_TRAIN["domain"].isin(source_domains)
    ].copy().reset_index(drop=True)

    if FULL_VAL is not None:
        train_df = source_train
        val_df = FULL_VAL[
            FULL_VAL["domain"].isin(source_domains)
        ].copy().reset_index(drop=True)
    else:
        train_df, val_df = source_only_validation_split(
            source_train,
            seed=42,
        )

    test_df = FULL_TEST[
        FULL_TEST["domain"].eq(target_domain)
    ].copy().reset_index(drop=True)

    assert target_domain not in set(train_df["domain"])
    assert target_domain not in set(val_df["domain"])
    assert set(test_df["domain"]) == {target_domain}

    train_fam = set(
        train_df["scenario_family_id"]
        .fillna(train_df["id"])
        .astype(str)
    )
    val_fam = set(
        val_df["scenario_family_id"]
        .fillna(val_df["id"])
        .astype(str)
    )
    test_fam = set(
        test_df["scenario_family_id"]
        .fillna(test_df["id"])
        .astype(str)
    )

    assert train_fam.isdisjoint(val_fam)
    assert train_fam.isdisjoint(test_fam)
    assert val_fam.isdisjoint(test_fam)

    for form in TEXT_FORMS:
        train_text = set(train_df[form].astype(str))
        assert train_text.isdisjoint(set(val_df[form].astype(str)))
        assert train_text.isdisjoint(set(test_df[form].astype(str)))

    # Smoke mode still runs all 3 LODO folds, but with a tiny subset.
    if SMOKE_RUN:
        train_df = smoke_sample(
            train_df,
            SMOKE_TRAIN_SCENARIOS,
            101,
        )
        val_df = smoke_sample(
            val_df,
            SMOKE_VAL_SCENARIOS,
            102,
        )
        test_df = smoke_sample(
            test_df,
            SMOKE_TEST_SCENARIOS,
            103,
        )

    return {
        "target": target_domain,
        "sources": source_domains,
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

FOLDS = {
    target: build_lodo_fold(target)
    for target in TARGET_DOMAINS
}

for target,fold in FOLDS.items():
    print(
        target,
        "| train:", len(fold["train"]),
        "| val:", len(fold["val"]),
        "| test:", len(fold["test"]),
        "| sources:", fold["sources"],
    )

print("LODO leakage checks: PASS")

In [ ]:
# ============================================================
# CELL 4 — PREPARE TEXT ROWS
# ============================================================
FORM_ORDER = {f:i for i,f in enumerate(TEXT_FORMS)}

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

def expand_rows(df, evaluation=False):
    x = df.copy()
    x["source_order"] = np.arange(len(x))
    x["gold_risk"] = x["decision"].ne("ALLOW").astype(int)

    base = [
        "id", "scenario_family_id", "domain",
        "security_intent", "hard_negative_type",
        "decision", "source_order", "gold_risk",
    ]

    if evaluation:
        base += ["in_decision_matched_test"]

    out = x.melt(
        id_vars=base,
        value_vars=TEXT_FORMS,
        var_name="text_form",
        value_name="text",
    )

    out["form_order"] = out["text_form"].map(FORM_ORDER)

    return (
        out.sort_values(
            ["source_order", "form_order"],
            kind="stable",
        )
        .reset_index(drop=True)
    )

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

def to_hf_dataset(df):
    ds = Dataset.from_pandas(
        df[["text", "gold_risk"]]
        .rename(columns={"gold_risk": "labels"}),
        preserve_index=False,
    )

    return ds.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"],
    )

def train_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)

    return {
        "macro_f1": f1_score(
            labels,
            pred,
            average="macro",
        ),
        "accuracy": accuracy_score(
            labels,
            pred,
        ),
    }

In [ ]:
# ============================================================
# CELL 5 — PREDICT + METRICS
# ============================================================
def predict_frame(model, df, target_domain, seed):
    model.eval()

    records = []
    device = next(model.parameters()).device

    for start in range(0, len(df), EVAL_BATCH_SIZE):
        batch = df.iloc[
            start:start+EVAL_BATCH_SIZE
        ]

        enc = tokenizer(
            batch["text"].astype(str).tolist(),
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            return_tensors="pt",
        )

        enc = {
            k:v.to(device)
            for k,v in enc.items()
        }

        with torch.inference_mode():
            logits = model(**enc).logits
            prob = torch.softmax(
                logits.float(),
                dim=-1,
            )[:,1].cpu().numpy()

        pred = (prob >= 0.5).astype(int)

        for (_,row),score,p in zip(
            batch.iterrows(),
            prob,
            pred,
        ):
            records.append({
                "id": row["id"],
                "scenario_family_id": row["scenario_family_id"],
                "target_domain": target_domain,
                "seed": seed,
                "domain": row["domain"],
                "text_form": row["text_form"],
                "decision": row["decision"],
                "security_intent": row["security_intent"],
                "hard_negative_type": row["hard_negative_type"],
                "in_decision_matched_test": bool(
                    row["in_decision_matched_test"]
                ),
                "gold_risk": int(row["gold_risk"]),
                "prediction": int(p),
                "risk_score": float(score),
            })

    return pd.DataFrame(records)

def hard_negative_mask(df):
    declared = (
        df["hard_negative_type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["", "none", "nan", "na", "n/a", "false", "0"])
        == False
    )

    return (
        declared
        & df["gold_risk"].eq(0)
    )

def compute_metrics(df):
    y = df["gold_risk"].astype(int).to_numpy()
    p = df["prediction"].astype(int).to_numpy()
    s = df["risk_score"].astype(float).to_numpy()

    hn = hard_negative_mask(df)

    out = {
        "n": len(df),
        "accuracy": accuracy_score(y,p),
        "balanced_accuracy": balanced_accuracy_score(y,p),
        "macro_f1": f1_score(
            y,p,
            average="macro",
            zero_division=0,
        ),
        "precision": precision_score(
            y,p,
            zero_division=0,
        ),
        "risk_recall": recall_score(
            y,p,
            zero_division=0,
        ),
        "specificity": recall_score(
            y,p,
            pos_label=0,
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(y,p),
        "hard_negative_over_refusal": (
            float(df.loc[hn, "prediction"].eq(1).mean())
            if hn.any()
            else np.nan
        ),
    }

    if len(np.unique(y)) == 2:
        out["auroc"] = roc_auc_score(y,s)
        out["auprc"] = average_precision_score(y,s)
    else:
        out["auroc"] = np.nan
        out["auprc"] = np.nan

    return out

In [ ]:
# ============================================================
# CELL 6 — RUN LODO: 3 DOMAINS × 3 SEEDS
# ============================================================
all_predictions = []
run_rows = []

for target_domain in TARGET_DOMAINS:
    fold = FOLDS[target_domain]

    train_rows = expand_rows(
        fold["train"],
        evaluation=False,
    )
    val_rows = expand_rows(
        fold["val"],
        evaluation=False,
    )
    test_rows = expand_rows(
        fold["test"],
        evaluation=True,
    )

    train_ds = to_hf_dataset(train_rows)
    val_ds = to_hf_dataset(val_rows)

    target_pred_dir = (
        PRED_DIR
        / target_domain.replace("-", "_")
    )
    target_pred_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for seed in SEEDS:
        print(
            f"\n=== TARGET {target_domain} | SEED {seed} ==="
        )

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        set_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=2,
            id2label={
                0:"benign",
                1:"security_risk",
            },
            label2id={
                "benign":0,
                "security_risk":1,
            },
        )

        steps_per_epoch = max(
            1,
            math.ceil(
                len(train_ds)
                / (
                    TRAIN_BATCH_SIZE
                    * GRADIENT_ACCUMULATION_STEPS
                )
            ),
        )

        warmup_steps = int(
            steps_per_epoch
            * ACTIVE_EPOCHS
            * WARMUP_RATIO
        )

        run_dir = (
            CKPT_DIR
            / target_domain.replace("-", "_")
            / f"seed_{seed}"
        )

        args_kwargs = {
            "output_dir": str(run_dir),
            "learning_rate": LEARNING_RATE,
            "per_device_train_batch_size": TRAIN_BATCH_SIZE,
            "per_device_eval_batch_size": EVAL_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "num_train_epochs": ACTIVE_EPOCHS,
            "weight_decay": WEIGHT_DECAY,
            "warmup_steps": warmup_steps,
            "save_strategy": "epoch",
            "logging_strategy": "epoch",
            "load_best_model_at_end": True,
            "metric_for_best_model": "macro_f1",
            "greater_is_better": True,
            "save_total_limit": 1,
            "fp16": torch.cuda.is_available(),
            "report_to": "none",
            "seed": seed,
            "data_seed": seed,
        }

        params = inspect.signature(
            TrainingArguments.__init__
        ).parameters

        if "eval_strategy" in params:
            args_kwargs["eval_strategy"] = "epoch"
        else:
            args_kwargs["evaluation_strategy"] = "epoch"

        args_kwargs = {
            k:v
            for k,v in args_kwargs.items()
            if k in params
        }

        trainer_kwargs = {
            "model": model,
            "args": TrainingArguments(
                **args_kwargs
            ),
            "train_dataset": train_ds,
            "eval_dataset": val_ds,
            "data_collator": DataCollatorWithPadding(tokenizer),
            "compute_metrics": train_metrics,
            "callbacks": [
                EarlyStoppingCallback(
                    early_stopping_patience=PATIENCE
                )
            ],
        }

        trainer_params = inspect.signature(
            Trainer.__init__
        ).parameters

        if "processing_class" in trainer_params:
            trainer_kwargs["processing_class"] = tokenizer
        elif "tokenizer" in trainer_params:
            trainer_kwargs["tokenizer"] = tokenizer

        trainer = Trainer(
            **trainer_kwargs
        )

        trainer.train()
        val_metrics = trainer.evaluate()

        pred_df = predict_frame(
            trainer.model,
            test_rows,
            target_domain,
            seed,
        )

        pred_path = (
            target_pred_dir
            / f"seed_{seed}.csv"
        )

        pred_df.to_csv(
            pred_path,
            index=False,
        )

        all_predictions.append(
            pred_df
        )

        run_rows.append({
            "target_domain": target_domain,
            "training_domains": " + ".join(
                fold["sources"]
            ),
            "seed": seed,
            "train_scenarios": len(fold["train"]),
            "val_scenarios": len(fold["val"]),
            "test_scenarios": len(fold["test"]),
            "val_macro_f1": val_metrics.get("eval_macro_f1"),
            "val_accuracy": val_metrics.get("eval_accuracy"),
        })

        del model, trainer
        shutil.rmtree(
            run_dir,
            ignore_errors=True,
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()

    del train_ds, val_ds
    gc.collect()

RUN_SUMMARY = pd.DataFrame(
    run_rows
)

RUN_SUMMARY.to_csv(
    METRIC_DIR / "training_runs.csv",
    index=False,
)

ALL_PRED = pd.concat(
    all_predictions,
    ignore_index=True,
)

ALL_PRED.to_csv(
    PRED_DIR / "all_lodo_predictions.csv",
    index=False,
)

print("\nLODO training complete.")

In [ ]:
# ============================================================
# CELL 7 — DOMAIN-WISE + PER-FORM + MATCHED RESULTS
# ============================================================
overall_rows = []
form_rows = []
matched_rows = []

for (
    target_domain,
    seed
), g in ALL_PRED.groupby(
    ["target_domain", "seed"],
    sort=False,
):
    overall_rows.append({
        "target_domain": target_domain,
        "seed": seed,
        **compute_metrics(g),
    })

    for form, sub in g.groupby(
        "text_form",
        sort=False,
    ):
        form_rows.append({
            "target_domain": target_domain,
            "seed": seed,
            "text_form": form,
            **compute_metrics(sub),
        })

    matched = g[
        g["in_decision_matched_test"]
    ]

    if len(matched):
        matched_rows.append({
            "target_domain": target_domain,
            "seed": seed,
            **compute_metrics(matched),
        })

PER_SEED = pd.DataFrame(
    overall_rows
)

PER_FORM = pd.DataFrame(
    form_rows
)

MATCHED = pd.DataFrame(
    matched_rows
)

PER_SEED.to_csv(
    METRIC_DIR / "lodo_per_seed.csv",
    index=False,
)

PER_FORM.to_csv(
    METRIC_DIR / "lodo_per_form.csv",
    index=False,
)

MATCHED.to_csv(
    METRIC_DIR / "lodo_decision_matched.csv",
    index=False,
)

DOMAIN_SUMMARY = (
    PER_SEED
    .groupby(
        "target_domain",
        as_index=False,
    )
    .agg(
        macro_f1_mean=("macro_f1","mean"),
        macro_f1_std=("macro_f1","std"),
        balanced_accuracy_mean=("balanced_accuracy","mean"),
        balanced_accuracy_std=("balanced_accuracy","std"),
        risk_recall_mean=("risk_recall","mean"),
        specificity_mean=("specificity","mean"),
        over_refusal_mean=("hard_negative_over_refusal","mean"),
        auroc_mean=("auroc","mean"),
        auprc_mean=("auprc","mean"),
    )
)

if len(MATCHED):
    matched_summary = (
        MATCHED
        .groupby(
            "target_domain",
            as_index=False,
        )
        .agg(
            decision_matched_macro_f1_mean=("macro_f1","mean"),
            decision_matched_macro_f1_std=("macro_f1","std"),
        )
    )

    DOMAIN_SUMMARY = (
        DOMAIN_SUMMARY
        .merge(
            matched_summary,
            on="target_domain",
            how="left",
        )
    )

FORM_SUMMARY = (
    PER_FORM
    .groupby(
        [
            "target_domain",
            "text_form",
        ],
        as_index=False,
    )
    .agg(
        macro_f1_mean=("macro_f1","mean"),
        macro_f1_std=("macro_f1","std"),
        balanced_accuracy_mean=("balanced_accuracy","mean"),
        risk_recall_mean=("risk_recall","mean"),
        specificity_mean=("specificity","mean"),
        over_refusal_mean=("hard_negative_over_refusal","mean"),
    )
)

DOMAIN_SUMMARY.to_csv(
    METRIC_DIR / "lodo_domain_summary.csv",
    index=False,
)

FORM_SUMMARY.to_csv(
    METRIC_DIR / "lodo_domain_by_form_summary.csv",
    index=False,
)

MACRO_AVERAGE = pd.DataFrame([{
    "macro_f1": DOMAIN_SUMMARY["macro_f1_mean"].mean(),
    "balanced_accuracy": DOMAIN_SUMMARY["balanced_accuracy_mean"].mean(),
    "risk_recall": DOMAIN_SUMMARY["risk_recall_mean"].mean(),
    "specificity": DOMAIN_SUMMARY["specificity_mean"].mean(),
    "hard_negative_over_refusal": DOMAIN_SUMMARY["over_refusal_mean"].mean(),
    "decision_matched_macro_f1": (
        DOMAIN_SUMMARY["decision_matched_macro_f1_mean"].mean()
        if "decision_matched_macro_f1_mean" in DOMAIN_SUMMARY
        else np.nan
    ),
}])

MACRO_AVERAGE.to_csv(
    METRIC_DIR / "lodo_macro_average.csv",
    index=False,
)

print("\nLODO DOMAIN SUMMARY")
display(
    DOMAIN_SUMMARY.round(4)
)

print("\nLODO DOMAIN × FORM")
display(
    FORM_SUMMARY.round(4)
)

print("\nLODO MACRO-AVERAGE")
display(
    MACRO_AVERAGE.round(4)
)

In [ ]:
# ============================================================
# CELL 8 — PACKAGE RESULTS
# ============================================================
archive = Path(
    shutil.make_archive(
        str(WORK_ROOT / f"xlmr_base_lodo_only_{RUN_MODE}_results"),
        "zip",
        OUTPUT_DIR,
    )
)

print("Results:", OUTPUT_DIR)
print("ZIP:", archive)

from IPython.display import FileLink
display(FileLink(str(archive)))